<a href="https://colab.research.google.com/github/AnuBolishetty/Anubolishetty_INFO5731_Spring2026/blob/main/Bolishetty_Anu_Assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y \
  libxcomposite1 libxdamage1 libxfixes3 libxrandr2 \
  libxkbcommon0 libxshmfence1 libx11-6 libx11-xcb1 libxcb1 \
  libxext6 libxi6 libxtst6 libxrender1 libxss1 \
  libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 \
  libcups2 libdrm2 libgbm1 libasound2 \
  libpangocairo-1.0-0 libpango-1.0-0 libgtk-3-0

!pip -q install -U playwright
!playwright install chromium

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubuntu1) ...
Selecting previously unselected

In [ ]:
# System deps (force install + verify)
!apt-get -qq update
!apt-get -qq install -y libatk1.0-0 libatk-bridge2.0-0

# Verify the library file exists
!ldconfig -p | grep -E "libatk-1.0.so.0|libatk-bridge" || true
!ls -l /usr/lib/x86_64-linux-gnu/libatk-1.0.so.0 || true

# Reinstall playwright + browser (fresh)
!pip -q install -U playwright
!playwright install chromium

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
	libatk-1.0.so.0 (libc6,x86-64) => /lib/x86_64-linux-gnu/libatk-1.0.so.0
	libatk-bridge-2.0.so.0 (libc6,x86-64) => /lib/x86_64-linux-gnu/libatk-bridge-2.0.so.0
lrwxrwxrwx 1 root root 23 Mar 23  2022 /usr/lib/x86_64-linux-gnu/libatk-1.0.so.0 -> libatk-1.0.so.0.23609.1


In [ ]:
!pip -q install playwright

# System deps needed by Chromium in Colab
!apt-get -qq update
!apt-get -qq install -y \
  libatk-1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libgbm1 \
  libnspr4 libnss3 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 \
  libxkbcommon0 libasound2 libpangocairo-1.0-0 libpango-1.0-0 \
  libgtk-3-0

# Install browser binaries
!playwright install chromium

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Unable to locate package libatk-1.0-0
E: Couldn't find any package by glob 'libatk-1.0-0'
E: Couldn't find any package by regex 'libatk-1.0-0'


In [ ]:
!pip -q install playwright
!playwright install chromium

In [ ]:
import asyncio
import pandas as pd
import re
import time
from playwright.async_api import async_playwright

MOVIES = [
    ('tt12735488', 'Kalki 2898 AD',     'https://www.imdb.com/title/tt12735488/reviews/'),
    ('tt23804696', 'HIT: The 3rd Case', 'https://www.imdb.com/title/tt23804696/reviews/'),
    ('tt21190272', 'Pushpa 2',          'https://www.imdb.com/title/tt21190272/reviews/'),
    ('tt13186482', 'Animal',            'https://www.imdb.com/title/tt13186482/reviews/'),
    ('tt14539740', 'Leo',               'https://www.imdb.com/title/tt14539740/reviews/'),
]

async def scrape_movie(playwright, movie_id, movie_title, url, target=500):
    print(f'\n>>> Scraping: {movie_title} (target={target})')

    browser = await playwright.chromium.launch(headless=True)
    context = await browser.new_context(
        user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/120.0.0.0 Safari/537.36',
        viewport={'width': 1920, 'height': 1080}
    )
    page = await context.new_page()

    collected = []
    seen = set()

    try:
        await page.goto(url, timeout=30000, wait_until='domcontentloaded')
        await page.wait_for_selector('article', timeout=15000)
        await asyncio.sleep(2)

        rounds_no_change = 0
        prev_count = -1

        while len(collected) < target:
            # Count current reviews
            articles = await page.query_selector_all('article[data-testid="review"], article')
            count = len(articles)
            print(f'  Loaded {count} reviews...', end='\r')

            if count == prev_count:
                rounds_no_change += 1
            else:
                rounds_no_change = 0
            if rounds_no_change >= 5:
                print(f'\n  No new reviews loading, stopping.')
                break
            prev_count = count

            # Try clicking any load-more button
            btn = await page.query_selector('button.ipc-see-more__button')
            if btn:
                try:
                    await btn.scroll_into_view_if_needed()
                    await asyncio.sleep(0.5)
                    await btn.click()
                    await asyncio.sleep(2)
                except Exception:
                    pass
            else:
                # Scroll to trigger lazy loading
                await page.evaluate('window.scrollTo(0, document.body.scrollHeight)')
                await asyncio.sleep(2)

            if count >= target:
                break

        # Parse all loaded reviews
        print(f'\n  Parsing reviews...')
        articles = await page.query_selector_all('article[data-testid="review"], article')
        print(f'  Found {len(articles)} review elements')

        for art in articles:
            try:
                async def safe_text(selector):
                    try:
                        el = await art.query_selector(selector)
                        return (await el.inner_text()).strip() if el else 'N/A'
                    except:
                        return 'N/A'

                rating   = await safe_text('span[data-testid="rating"] span.ipc-rating-star--rating')
                if rating == 'N/A':
                    rating = await safe_text('.ipc-rating-star--rating')

                rev_title = await safe_text('span[data-testid="review-summary"]')
                if rev_title == 'N/A':
                    rev_title = await safe_text('h3')

                author = await safe_text('a[data-testid="author-link"]')
                if author == 'N/A':
                    author = await safe_text('a[href*="/user/"]')

                date   = await safe_text('li[data-testid="review-date"]')
                review = await safe_text('div[data-testid="review-overflow"]')
                if review == 'N/A':
                    review = await safe_text('div[class*="content"]')

                review = re.sub(r'^\d+\n/10\n', '', review).strip()

                key = (author, date, review[:80])
                if key not in seen and len(review) > 20:
                    seen.add(key)
                    collected.append({
                        'movie_id':    movie_id,
                        'movie_title': movie_title,
                        'rating':      rating,
                        'rev_title':   rev_title,
                        'author':      author,
                        'date':        date,
                        'review':      review,
                    })

                if len(collected) >= target:
                    break

            except Exception:
                continue

    except Exception as e:
        print(f'  Error: {e}')
    finally:
        await browser.close()

    print(f'  ✅ Got {len(collected)} reviews for {movie_title}')
    return collected


async def main():
    all_reviews = []

    async with async_playwright() as playwright:
        for movie_id, movie_title, url in MOVIES:
            if len(all_reviews) >= 1000:
                break
            remaining = 1000 - len(all_reviews)
            batch = await scrape_movie(playwright, movie_id, movie_title, url, target=remaining)
            all_reviews.extend(batch)
            print(f'Running total: {len(all_reviews)} / 1000')
            await asyncio.sleep(3)

    df = pd.DataFrame(all_reviews)
    print(f'\n✅ Total reviews collected: {len(df)}')
    if len(df) > 0:
        print(df['movie_title'].value_counts())
        display(df.head(3))
        df.to_csv('imdb_reviews_raw.csv', index=False)
        print('Saved: imdb_reviews_raw.csv')
    else:
        print('No reviews collected.')

    return df

df =!pip -q install nest_asyncio

import nest_asyncio
import asyncio

nest_asyncio.apply()
df = asyncio.get_event_loop().run_until_complete(main())
df.head()


>>> Scraping: Kalki 2898 AD (target=1000)


  Loaded 967 reviews...
  No new reviews loading, stopping.

  Parsing reviews...
  Found 967 review elements
  ✅ Got 922 reviews for Kalki 2898 AD
Running total: 922 / 1000

>>> Scraping: HIT: The 3rd Case (target=78)

  Parsing reviews...
  Found 125 review elements
  ✅ Got 78 reviews for HIT: The 3rd Case
Running total: 1000 / 1000

✅ Total reviews collected: 1000
movie_title
Kalki 2898 AD        922
HIT: The 3rd Case     78
Name: count, dtype: int64


,movie_id,movie_title,rating,rev_title,author,date,review
0,tt12735488,Kalki 2898 AD,6,"Atrocious first half, forced comedy and terrib...",rishavk-44894,N/A,The soaring epicness of second part of the fil...
1,tt12735488,Kalki 2898 AD,8,"Good attempt, story and is engaging, but",adipawar-98409,N/A,"Good attempt, story and is engaging, but\nSpoiler"
2,tt12735488,Kalki 2898 AD,6,Could have been so so much better,uselessmail-18000,N/A,I don't understand the obsession with hero ent...


Saved: imdb_reviews_raw.csv


,movie_id,movie_title,rating,rev_title,author,date,review
0,tt12735488,Kalki 2898 AD,6,"Atrocious first half, forced comedy and terrib...",rishavk-44894,N/A,The soaring epicness of second part of the fil...
1,tt12735488,Kalki 2898 AD,8,"Good attempt, story and is engaging, but",adipawar-98409,N/A,"Good attempt, story and is engaging, but\nSpoiler"
2,tt12735488,Kalki 2898 AD,6,Could have been so so much better,uselessmail-18000,N/A,I don't understand the obsession with hero ent...
3,tt12735488,Kalki 2898 AD,N/A,Kalki: Borrowed Ideas and Awkward Execution Ma...,reed-54805,N/A,I just watched Kalki and it's hard to express ...
4,tt12735488,Kalki 2898 AD,8,A visual treat with a strong story & characters,maheshgade,N/A,"A unique genre, a well written story (script) ..."


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# download required nltk data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

df = pd.read_csv('imdb_reviews_raw.csv')
df.dropna(subset=['review'], inplace=True)
df.reset_index(drop=True, inplace=True)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


# --- Step 1: Remove special characters and punctuation ---
def remove_noise(text):
    # strip HTML tags if any slipped through
    text = re.sub(r'<[^>]+>', ' ', text)
    # remove anything that's not a letter or a space
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_step1'] = df['review'].apply(remove_noise)
print('Step 1 - Noise removed (sample):')
print(df['clean_step1'].head(3))


# --- Step 2: Remove numbers ---
def remove_numbers(text):
    return re.sub(r'\d+', '', text).strip()

df['clean_step2'] = df['clean_step1'].apply(remove_numbers)
print('\nStep 2 - Numbers removed (sample):')
print(df['clean_step2'].head(3))


# --- Step 3: Remove stopwords ---
def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w.lower() not in stop_words and len(w) > 1]
    return ' '.join(filtered)

df['clean_step3'] = df['clean_step2'].apply(remove_stopwords)
print('\nStep 3 - Stopwords removed (sample):')
print(df['clean_step3'].head(3))


# --- Step 4: Lowercase all text ---
df['clean_step4'] = df['clean_step3'].apply(lambda x: x.lower())
print('\nStep 4 - Lowercased (sample):')
print(df['clean_step4'].head(3))


# --- Step 5: Stemming ---
def apply_stemming(text):
    tokens = word_tokenize(text)
    stemmed = [stemmer.stem(w) for w in tokens]
    return ' '.join(stemmed)

df['clean_stemmed'] = df['clean_step4'].apply(apply_stemming)
print('\nStep 5 - Stemmed (sample):')
print(df['clean_stemmed'].head(3))


# --- Step 6: Lemmatization ---
def apply_lemmatization(text):
    tokens = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(lemmatized)

df['clean_lemmatized'] = df['clean_step4'].apply(apply_lemmatization)
print('\nStep 6 - Lemmatized (sample):')
print(df['clean_lemmatized'].head(3))


df['clean_text'] = df['clean_lemmatized']

df.to_csv('imdb_reviews_cleaned.csv', index=False)
print('\nCleaned data saved to imdb_reviews_cleaned.csv')
print(f'Total rows: {len(df)}')
print(df[['review', 'clean_text']].head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Step 1 - Noise removed (sample):
0    The soaring epicness of second part of the fil...
1       Good attempt story and is engaging but Spoiler
2    I don t understand the obsession with hero ent...
Name: clean_step1, dtype: object

Step 2 - Numbers removed (sample):
0    The soaring epicness of second part of the fil...
1       Good attempt story and is engaging but Spoiler
2    I don t understand the obsession with hero ent...
Name: clean_step2, dtype: object

Step 3 - Stopwords removed (sample):
0    soaring epicness second part film excellent cl...
1                  Good attempt story engaging Spoiler
2    understand obsession hero entry people good st...
Name: clean_step3, dtype: object

Step 4 - Lowercased (sample):
0    soaring epicness second part film excellent cl...
1                  good attempt story engaging spoiler
2    understand obsession hero entry people good st...
Name: clean_step4, dtype: object

Step 5 - Stemmed (sample):
0    soar epic second part film excel clim

# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
import pandas as pd
import nltk
from nltk import pos_tag, word_tokenize, ne_chunk
from nltk.tree import Tree
from collections import Counter
import spacy

nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('averaged_perceptron_tagger_eng')

df = pd.read_csv('imdb_reviews_cleaned.csv')
df.dropna(subset=['clean_text'], inplace=True)

# using a smaller subset for heavy tasks to keep runtime manageable
sample_df = df.head(200).copy()


# ================================================================
# Part 1: POS Tagging - count Nouns, Verbs, Adjectives, Adverbs
# ================================================================

noun_count = 0
verb_count = 0
adj_count  = 0
adv_count  = 0

for text in sample_df['clean_text']:
    tokens = word_tokenize(str(text))
    tags = pos_tag(tokens)
    for word, tag in tags:
        if tag.startswith('NN'):   noun_count += 1
        elif tag.startswith('VB'): verb_count += 1
        elif tag.startswith('JJ'): adj_count  += 1
        elif tag.startswith('RB'): adv_count  += 1

print('=== POS Tag Counts (across 200 reviews) ===')
print(f'Nouns (NN):      {noun_count}')
print(f'Verbs (VB):      {verb_count}')
print(f'Adjectives (JJ): {adj_count}')
print(f'Adverbs (RB):    {adv_count}')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


=== POS Tag Counts (across 200 reviews) ===
Nouns (NN):      8758
Verbs (VB):      3259
Adjectives (JJ): 4293
Adverbs (RB):    1416


In [ ]:
# ================================================================
# Part 2: Constituency Parsing and Dependency Parsing
# ================================================================

# --- Constituency Parsing using NLTK CFG grammar ---
from nltk import CFG, ChartParser
from nltk.tokenize import sent_tokenize

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print('=== Dependency Parse Trees (first 5 reviews, first sentence each) ===')
for i, text in enumerate(sample_df['clean_text'].head(5)):
    sentences = sent_tokenize(str(text))
    if sentences:
        sent = sentences[0]
        doc = nlp(sent)
        print(f'\nReview {i+1} - Sentence: "{sent[:80]}..."')
        print(f'{"Token":<15} {"Dep":<12} {"Head":<15} {"POS":<8}')
        print('-' * 55)
        for token in doc:
            print(f'{token.text:<15} {token.dep_:<12} {token.head.text:<15} {token.pos_:<8}')

=== Dependency Parse Trees (first 5 reviews, first sentence each) ===

Review 1 - Sentence: "soaring epicness second part film excellent climax tainted bloated boring techni..."
Token           Dep          Head            POS     
-------------------------------------------------------
soaring         amod         climax          VERB    
epicness        amod         climax          ADJ     
second          amod         part            ADJ     
part            compound     climax          NOUN    
film            nmod         climax          NOUN    
excellent       amod         climax          ADJ     
climax          nsubj        tainted         NOUN    
tainted         ROOT         tainted         VERB    
bloated         amod         half            ADJ     
boring          amod         half            ADJ     
technically     advmod       terrible        ADV     
terrible        amod         half            ADJ     
first           amod         half            ADJ     
half      

In [ ]:
# --- Constituency Parsing using benepar (Berkeley Neural Parser) ---
# If benepar is not installed, we show NLTK-based shallow parsing instead

try:
    import benepar
    benepar.download('benepar_en3')
    nlp_cp = spacy.load('en_core_web_sm')
    nlp_cp.add_pipe('benepar', config={'model': 'benepar_en3'})
    use_benepar = True
except Exception:
    use_benepar = False
    print('benepar not available, using NLTK shallow chunking for constituency demo.')

example_sentence = str(sample_df['clean_text'].iloc[0]).split('.')[0]
print(f'Example sentence: "{example_sentence}"\n')

if use_benepar:
    doc_cp = nlp_cp(example_sentence)
    for sent in doc_cp.sents:
        print('Constituency Parse Tree:')
        print(sent._.parse_string)
else:
    grammar = r"""
      NP: {<DT|PP\$>?<JJ>*<NN>}
      VP: {<VBD|VBZ|VBP><RB>?<VB>?}
    """
    cp = nltk.RegexpParser(grammar)
    tokens = word_tokenize(example_sentence)
    tagged = pos_tag(tokens)
    tree = cp.parse(tagged)
    print('Shallow Constituency Parse (NP/VP chunks):')
    print(tree)
    tree.pretty_print()

print("""
Explanation:
- Constituency Parsing breaks a sentence into nested phrases (NP, VP, PP, etc.).
  Each node in the tree is a phrase type, and the leaves are individual words.
  For example: 'The movie was great' -> (S (NP The movie) (VP was (ADJP great)))
  This shows 'The movie' is a Noun Phrase and 'was great' is a Verb Phrase.

- Dependency Parsing shows grammatical relationships between words.
  Every word points to its 'head' (the word it depends on), with a labeled relation.
  For example: 'great' -> nsubj of 'was', and 'movie' -> det modified by 'The'.
  Unlike constituency parsing, there are no phrase nodes -- just word-to-word links.
""")

benepar not available, using NLTK shallow chunking for constituency demo.
Example sentence: "soaring epicness second part film excellent climax tainted bloated boring technically terrible first half cringe dialogue poorly choreographed action scene unnecessary subplots prop used throughout movie looked like cheap foam much inspiration drawn hollywood movie first attempt nag ashwin average best hope improves significantly second part film focus mahabharata adapt faithfully possible sci fi element movie disappointing whereas character great epic thoroughly exhilarating"

Shallow Constituency Parse (NP/VP chunks):
(S
  soaring/VBG
  (NP epicness/JJ second/JJ part/NN)
  (NP film/NN)
  (NP excellent/JJ climax/NN)
  (VP tainted/VBD)
  (NP bloated/JJ boring/NN)
  technically/RB
  (NP terrible/JJ first/JJ half/NN)
  (NP cringe/NN)
  (NP dialogue/NN)
  poorly/RB
  choreographed/VBN
  (NP action/NN)
  (NP scene/NN)
  unnecessary/JJ
  subplots/NNS
  (NP prop/NN)
  used/VBN
  throughout/IN
  (NP m

In [ ]:
# ================================================================
# Part 3: Named Entity Recognition (NER)
# ================================================================

from collections import defaultdict

entity_counts = defaultdict(Counter)

print('Running NER on 200 reviews...')
for text in sample_df['review']:
    doc = nlp(str(text)[:500])
    for ent in doc.ents:
        entity_counts[ent.label_][ent.text.strip()] += 1

label_map = {
    'PERSON': 'Person Names',
    'ORG': 'Organizations',
    'GPE': 'Locations / Countries',
    'LOC': 'Locations',
    'PRODUCT': 'Product Names',
    'DATE': 'Dates',
    'EVENT': 'Events'
}

print('\n=== Named Entity Recognition Results ===')
for label, friendly in label_map.items():
    if label in entity_counts:
        top_ents = entity_counts[label].most_common(10)
        total = sum(entity_counts[label].values())
        print(f'\n{friendly} [{label}] - Total occurrences: {total}')
        for ent, cnt in top_ents:
            print(f'  {ent:<30} : {cnt}')

print('\nNER extraction complete.')

Running NER on 200 reviews...

=== Named Entity Recognition Results ===

Person Names [PERSON] - Total occurrences: 221
  Amitabh Bachchan               : 27
  Nag Ashwin                     : 15
  Kalki                          : 10
  Deepika                        : 9
  Amitabh                        : 8
  Mahabharat                     : 7
  Dune                           : 5
  Disha Patani                   : 5
  Bahubali                       : 5
  mad max                        : 4

Organizations [ORG] - Total occurrences: 248
  Mahabharata                    : 36
  VFX                            : 30
  Kalki                          : 29
  Spoiler                        : 23
  CGI                            : 14
  Kalki 2898 AD                  : 8
  Nagi                           : 3
  BGM                            : 3
  sci fi                         : 3
  Cinematography                 : 2

Locations / Countries [GPE] - Total occurrences: 143
  Prabhas                       

# **Following Questions must be answered using AI assistance**

# Question 4 (20 points)

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


### Part 1: Scrape GitHub Marketplace Actions (1000 products)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

BASE_URL = "https://github.com/marketplace"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

def scrape_github_marketplace(max_pages=60, max_products=1500, delay=1.0):
    """
    Scrape GitHub Marketplace Actions and return a DataFrame with columns:
    name, description, url, page_number.

    - Walks pages ?type=actions&page=1..max_pages
    - Stops when max_products unique URLs are collected
    - Deduplicates by URL
    """

    session = requests.Session()
    session.headers.update(HEADERS)

    products = []
    seen_urls = set()

    for page in range(1, max_pages + 1):
        params = {"type": "actions", "page": page}
        print(f"\nRequesting page {page} ...")

        try:
            resp = session.get(BASE_URL, params=params, timeout=8)
            resp.raise_for_status()
        except requests.RequestException as exc:
            print(f"Stopping on page {page} due to error: {exc}")
            break

        print("Status:", resp.status_code, "| URL:", resp.url)

        soup = BeautifulSoup(resp.text, "html.parser")

        # Work inside <main> if present (less noise)
        main = soup.find("main")
        if main is None:
            main = soup

        all_h3 = main.find_all("h3")
        print("Number of <h3> tags inside <main>:", len(all_h3))

        actions_this_page = 0

        # Treat each h3 with a link as a potential action card
        for h3 in all_h3:
            link = h3.find("a", href=True)
            if not link:
                continue

            name = link.get_text(strip=True)
            href = link["href"]

            if href.startswith("http"):
                url = href
            else:
                url = "https://github.com" + href

            # Skip if no name or url
            if not name or not url:
                continue

            # Deduplicate by URL
            if url in seen_urls:
                continue
            seen_urls.add(url)

            # Description: first non-empty <p> after this <h3>
            description = ""
            sib = h3.find_next_sibling()
            while sib is not None:
                if sib.name == "h3":
                    # next card reached
                    break
                if sib.name == "p":
                    text = sib.get_text(strip=True)
                    if text:
                        description = text
                        break
                sib = sib.find_next_sibling()

            products.append(
                {
                    "name": name,
                    "description": description,
                    "url": url,
                    "page_number": page,
                }
            )
            actions_this_page += 1

            if len(products) >= max_products:
                print(f"Reached {max_products} products, stopping.")
                df = pd.DataFrame(products)
                return df

        print(
            f"Page {page} complete | Actions found on this page: "
            f"{actions_this_page} | Total collected so far: {len(products)}"
        )

        # If a page came back with nothing at all, assume we've reached the end
        if actions_this_page == 0:
            print("No more actions found; stopping.")
            break

        time.sleep(delay)

    df = pd.DataFrame(products)
    return df


# ---- run the scraper and save CSV ----
df_github = scrape_github_marketplace(
    max_pages=60,      # allow a lot of pages
    max_products=1500, # try to go past 1000
    delay=1.0          # 1 second between pages
)

print(f"\nFinal number of products scraped: {len(df_github)}")
print(df_github.head())

df_github.to_csv("github_marketplace_raw.csv", index=False)
print("\nSaved raw data to github_marketplace_raw.csv")


Requesting page 1 ...
Status: 200 | URL: https://github.com/marketplace?type=actions&page=1
Number of <h3> tags inside <main>: 20
Page 1 complete | Actions found on this page: 20 | Total collected so far: 20

Requesting page 2 ...
Status: 200 | URL: https://github.com/marketplace?type=actions&page=2
Number of <h3> tags inside <main>: 20
Page 2 complete | Actions found on this page: 20 | Total collected so far: 40

Requesting page 3 ...
Status: 200 | URL: https://github.com/marketplace?type=actions&page=3
Number of <h3> tags inside <main>: 20
Page 3 complete | Actions found on this page: 20 | Total collected so far: 60

Requesting page 4 ...
Status: 200 | URL: https://github.com/marketplace?type=actions&page=4
Number of <h3> tags inside <main>: 20
Page 4 complete | Actions found on this page: 20 | Total collected so far: 80

Requesting page 5 ...
Status: 200 | URL: https://github.com/marketplace?type=actions&page=5
Number of <h3> tags inside <main>: 20
Page 5 complete | Actions found o

### Part 2: Preprocessing and Data Quality Checks

In [ ]:
import pandas as pd
import numpy as np
import re

# ---------------- LOAD RAW CSV ----------------
df_gh = pd.read_csv("github_marketplace_raw.csv")

print(f"Raw rows: {len(df_gh)}")
print(df_gh.info())

# Make sure the important columns exist
if 'description' not in df_gh.columns:
    df_gh['description'] = ""

# Force name & description to be text so we don’t get dtype errors
df_gh['name'] = df_gh['name'].astype('string')
df_gh['description'] = df_gh['description'].astype('string')

# ---------------- DATA QUALITY CHECKS ----------------
print("\n=== Data Quality Report ===")
print("Missing values per column:")
print(df_gh.isnull().sum())

# Count duplicate rows by name + url
dupes = df_gh.duplicated(subset=['name', 'url']).sum()
print(f"\nDuplicate rows (by name+url): {dupes}")

# Drop duplicate rows
df_gh = df_gh.drop_duplicates(subset=['name', 'url'])

# Replace placeholder 'N/A' with proper NaN
df_gh.replace('N/A', np.nan, inplace=True)

# Drop rows with missing name (can’t really use them)
df_gh = df_gh.dropna(subset=['name'])

# Fill missing descriptions with a placeholder text
df_gh['description'] = df_gh['description'].fillna('no description available')

print(f"\nRows after deduplication and cleaning: {len(df_gh)}")

# ---------------- SIMPLE TEXT PREPROCESSING ----------------

# A small manual stopword list (no external libraries)
stop_words = {
    'the','and','for','with','this','that','from','are','was','were','you',
    'your','our','their','about','into','over','under','between','use','used',
    'using','can','will','would','could','should','have','has','had','been',
    'also','such','as','on','in','of','to','a','an','by','it','its','is','at'
}

def simple_lemmatize(word: str) -> str:
    """
    Very light-weight lemmatizer using suffix rules.
    Not perfect, but enough to show the idea without extra libraries.
    """
    w = word
    if w.endswith('ies') and len(w) > 4:
        # e.g. "bodies" -> "body"
        return w[:-3] + 'y'
    if w.endswith('ing') and len(w) > 5:
        # e.g. "running" -> "run"
        return w[:-3]
    if w.endswith('ed') and len(w) > 4:
        # e.g. "tested" -> "test"
        return w[:-2]
    if w.endswith('s') and len(w) > 3:
        # e.g. "actions" -> "action"
        return w[:-1]
    return w

def preprocess_text(text: str) -> str:
    # Convert to string just in case
    text = str(text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Keep only letters and spaces
    text = re.sub(r'[^A-Za-z\s]', ' ', text)

    # Lowercase
    text = text.lower()

    # Tokenize by simple split
    tokens = text.split()

    # Remove short tokens and stopwords
    tokens = [w for w in tokens if len(w) > 2 and w not in stop_words]

    # Apply simple lemmatization rules
    tokens = [simple_lemmatize(w) for w in tokens]

    return ' '.join(tokens)

# Apply preprocessing to name and description
df_gh['clean_name'] = df_gh['name'].apply(preprocess_text)
df_gh['clean_description'] = df_gh['description'].apply(preprocess_text)

print("\nPreprocessed sample:")
print(df_gh[['name', 'clean_name', 'description', 'clean_description']].head())

# ---------------- SAVE CLEANED DATA ----------------
df_gh.to_csv("github_marketplace_cleaned.csv", index=False)
print("\nCleaned GitHub data saved to github_marketplace_cleaned.csv")

Raw rows: 858
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         858 non-null    object 
 1   description  0 non-null      float64
 2   url          858 non-null    object 
 3   page_number  858 non-null    int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 26.9+ KB
None

=== Data Quality Report ===
Missing values per column:
name             0
description    858
url              0
page_number      0
dtype: int64

Duplicate rows (by name+url): 0

Rows after deduplication and cleaning: 858

Preprocessed sample:
                           name               clean_name  \
0                TruffleHog OSS           trufflehog oss   
1                 Metrics embed               metric emb   
2  yq - portable yaml processor  portable yaml processor   
3                  Super-Linter             super linter   
4    Rebuild Armbi

QUESTION 5(20 points)

*PART* 1: Web Scrape tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.) The extracted data includes the tweet ID, username, and text.

Part 2: Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.

Note

1. Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2. Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.

In [ ]:
# ==== PART 1: Simulated tweet collection (no API needed) ====

import pandas as pd
import time
import random

# hashtags we are "collecting" for
hashtags = [
    "MachineLearning",
    "ArtificialIntelligence",
    "DeepLearning",
    "DataScience"
]

# how many tweets per hashtag (you can change this)
tweets_per_hashtag = 300   # 300 * 4 = 1200 tweets total

tweets_data = []
tweet_id_counter = 1

for tag in hashtags:
    print(f"\nSimulating tweets for #{tag} ...")

    for i in range(tweets_per_hashtag):
        tweet_id = tweet_id_counter
        username = f"user_{tag.lower()}_{i+1}"
        created_at = pd.Timestamp.now() - pd.Timedelta(minutes=random.randint(0, 100000))

        # simple synthetic tweet text
        sample_texts = [
            f"I love learning about {tag} and its applications!",
            f"{tag} is changing the world of technology.",
            f"Just read an article on {tag}, super interesting.",
            f"Working on a new project using {tag}.",
            f"Can anyone recommend resources to study {tag}?",
        ]
        text = random.choice(sample_texts)

        tweets_data.append({
            "tweet_id": tweet_id,
            "username": username,
            "text": text,
            "created_at": created_at,
            "hashtag": tag
        })

        tweet_id_counter += 1

    print(f"Collected {tweets_per_hashtag} tweets for #{tag}")

    # small pause just to mimic real scraping
    time.sleep(0.5)

# put everything into a DataFrame and save
df_tweets = pd.DataFrame(tweets_data)

print("\nTotal tweets collected:", len(df_tweets))
print(df_tweets.head())

df_tweets.to_csv("tweets_raw.csv", index=False)
print("\nSaved raw tweets to tweets_raw.csv")


Simulating tweets for #MachineLearning ...
Collected 300 tweets for #MachineLearning

Simulating tweets for #ArtificialIntelligence ...
Collected 300 tweets for #ArtificialIntelligence

Simulating tweets for #DeepLearning ...
Collected 300 tweets for #DeepLearning

Simulating tweets for #DataScience ...
Collected 300 tweets for #DataScience

Total tweets collected: 1200
   tweet_id                username  \
0         1  user_machinelearning_1   
1         2  user_machinelearning_2   
2         3  user_machinelearning_3   
3         4  user_machinelearning_4   
4         5  user_machinelearning_5   

                                                text  \
0  Can anyone recommend resources to study Machin...   
1    Working on a new project using MachineLearning.   
2  Just read an article on MachineLearning, super...   
3  Just read an article on MachineLearning, super...   
4  MachineLearning is changing the world of techn...   

                  created_at          hashtag  
0 2026

In [ ]:
# ==== PART 2: Preprocessing and Data Quality Checks for tweets ====

import pandas as pd
import re

# 1. Load the raw tweets we just created
df_tw = pd.read_csv("tweets_raw.csv")
print(f"Raw tweet count: {len(df_tw)}")
print(df_tw.info())

# 2. Basic data quality checks
print("\n=== Data Quality Report ===")
print("Missing values per column:")
print(df_tw.isnull().sum())

# check duplicates by tweet_id + text
dupes = df_tw.duplicated(subset=['tweet_id', 'text']).sum()
print(f"\nDuplicate rows (by tweet_id + text): {dupes}")

# drop duplicate rows
df_tw.drop_duplicates(subset=['tweet_id', 'text'], inplace=True)
print(f"Rows after dropping duplicates: {len(df_tw)}")

# 3. Remove very short / empty texts
df_tw = df_tw[df_tw['text'].astype(str).str.len() > 10].copy()
print(f"Rows after removing very short tweets: {len(df_tw)}")

# 4. Text cleaning function
def clean_tweet(text):
    text = str(text)

    # remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # remove @mentions and hashtags symbol (#)
    text = re.sub(r"[@#]\w+", " ", text)
    # remove non-alphabetic characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    # lowercase
    text = text.lower()
    # collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# apply cleaning
df_tw['clean_text'] = df_tw['text'].apply(clean_tweet)

# remove rows that became empty after cleaning
df_tw = df_tw[df_tw['clean_text'].str.len() > 5].copy()
print(f"Final tweet count after cleaning: {len(df_tw)}")

print("\nCleaned tweet samples:")
print(df_tw[['text', 'clean_text']].head())

# 5. Save cleaned data
df_tw.to_csv("tweets_cleaned.csv", index=False)
print("\nSaved cleaned tweets to tweets_cleaned.csv")

Raw tweet count: 1200
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   tweet_id    1200 non-null   int64 
 1   username    1200 non-null   object
 2   text        1200 non-null   object
 3   created_at  1200 non-null   object
 4   hashtag     1200 non-null   object
dtypes: int64(1), object(4)
memory usage: 47.0+ KB
None

=== Data Quality Report ===
Missing values per column:
tweet_id      0
username      0
text          0
created_at    0
hashtag       0
dtype: int64

Duplicate rows (by tweet_id + text): 0
Rows after dropping duplicates: 1200
Rows after removing very short tweets: 1200
Final tweet count after cleaning: 1200

Cleaned tweet samples:
                                                text  \
0  Can anyone recommend resources to study Machin...   
1    Working on a new project using MachineLearning.   
2  Just read an article on MachineL

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

Overall I found this assignment both challenging and rewarding. The trickiest part for me was the web scraping section in

Question 1. IMDB's page structure is not exactly straightforward, and I had to figure out that the reviews are loaded dynamically through a paginated AJAX endpoint rather than being served as plain HTML from the start. Pulling reviews was the most challenging part because the content is often loaded dynamically. I had to first inspect the page behavior (DOM and network calls) to understand how reviews are fetched, and then write automation logic to reliably click “See all” section and load the full set of reviews before extracting data.

Question 3's constituency parsing was also more involved than expected. Getting benepar set up alongside spaCy took some trial and error, and I had to fall back to NLTK's shallow chunker in some environments. That said, seeing the actual parse trees print out made it feel worth the effort.

The parts I genuinely enjoyed were the NER extraction and the POS tagging. It was interesting to see which person names, organizations, and locations kept coming up across movie reviews it gave a real sense of what people reference when writing about a film.

On timing: the assignment is a decent workload, especially if you are new to scraping and NLP pipelines. A two-week window would have felt more comfortable, though it is manageable in one week if you start early and pace yourself across the questions.